In [2]:
import pandas as pd
import json
import requests

# Option 1: Load from file
with open('aave.json', 'r') as f:
    data = json.load(f)

# Option 2: Fetch directly from API
# response = requests.get('https://api.llama.fi/protocol/aave-v3')
# data = response.json()

# Extract basic protocol information
protocol_info = {
    'id': data['id'],
    'name': data['name'],
    'url': data['url'],
    'description': data['description'],
    'gecko_id': data.get('gecko_id'),
    'cmcId': data.get('cmcId'),
    'twitter': data.get('twitter'),
    'symbol': data.get('symbol'),
    'address': data.get('address')
}

# Create a DataFrame for protocol metadata
df_protocol = pd.DataFrame([protocol_info])


In [3]:
df_protocol

,id,name,url,description,gecko_id,cmcId,twitter,symbol,address
0,parent#aave,Aave,https://aave.com,Aave is an Open Source and Non-Custodial proto...,aave,7278,aave,AAVE,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9


In [4]:
import pandas as pd
import json
import requests

# Option 1: Load from file
with open('aave.json', 'r') as f:
    data = json.load(f)

# Option 2: Fetch directly from API
# response = requests.get('https://api.llama.fi/protocol/aave-v3')
# data = response.json()

# Extract ALL data from chainTvls branch
chain_tvls_data = []

for chain_key, chain_data in data.get('chainTvls', {}).items():
    # chain_data is a dictionary that may contain multiple keys like 'tvl', 'tokensInUsd', 'tokens', etc.
    
    # Check if there's a 'tvl' array
    if 'tvl' in chain_data and isinstance(chain_data['tvl'], list):
        for entry in chain_data['tvl']:
            chain_tvls_data.append({
                'chain': chain_key,
                'metric_type': 'tvl',
                'date': pd.to_datetime(entry['date'], unit='s'),
                'totalLiquidityUSD': entry.get('totalLiquidityUSD')
            })
    
    # Check if there are other arrays (tokensInUsd, tokens, etc.)
    for key, value in chain_data.items():
        if key != 'tvl' and isinstance(value, list):
            for entry in value:
                record = {
                    'chain': chain_key,
                    'metric_type': key,
                    'date': pd.to_datetime(entry.get('date'), unit='s') if 'date' in entry else None
                }
                # Add all other fields from the entry
                for field, field_value in entry.items():
                    if field != 'date':
                        record[field] = field_value
                chain_tvls_data.append(record)

# Create DataFrame with all chainTvls data
df_chain_tvls = pd.DataFrame(chain_tvls_data)


In [5]:
df_chain_tvls

,chain,metric_type,date,totalLiquidityUSD
0,Ethereum-staking,tvl,2020-12-03 00:00:00,244094753
1,Ethereum-staking,tvl,2020-12-04 00:00:00,243189473
2,Ethereum-staking,tvl,2020-12-05 00:00:00,250327449
3,Ethereum-staking,tvl,2020-12-06 00:00:00,253742514
4,Ethereum-staking,tvl,2020-12-07 00:00:00,256062919
...,...,...,...,...
43314,Aptos-borrowed,tvl,2026-01-07 00:00:00,15965898
43315,Aptos-borrowed,tvl,2026-01-08 00:00:00,15977650
43316,Aptos-borrowed,tvl,2026-01-09 00:00:00,15998925
43317,Aptos-borrowed,tvl,2026-01-10 00:00:00,16014125


In [7]:
#df_chain_tvls['chain'].value_counts()
df_chain_tvls['date'].max()

Timestamp('2026-01-10 15:28:59')

In [ ]:
https://api.llama.fi/protocol/aave

In [4]:
import pandas as pd
import json
import requests

response = requests.get('https://api.llama.fi/protocol/aave')
data = response.json()

chain_tvls_data = []

for chain_key, chain_data in data.get('chainTvls', {}).items():
    if 'tvl' in chain_data and isinstance(chain_data['tvl'], list):
        for entry in chain_data['tvl']:
            chain_tvls_data.append({
                'chain': chain_key,
                'metric_type': 'tvl',
                'date': pd.to_datetime(entry['date'], unit='s'),
                'totalLiquidityUSD': entry.get('totalLiquidityUSD')
            })
    
    for key, value in chain_data.items():
        if key != 'tvl' and isinstance(value, list):
            for entry in value:
                record = {
                    'chain': chain_key,
                    'metric_type': key,
                    'date': pd.to_datetime(entry.get('date'), unit='s') if 'date' in entry else None
                }
                for field, field_value in entry.items():
                    if field != 'date':
                        record[field] = field_value
                chain_tvls_data.append(record)

df_chain_tvls = pd.DataFrame(chain_tvls_data)

df_filtered = df_chain_tvls[df_chain_tvls['date'] > '2025-01-01']


In [7]:
df_chain_tvls

,chain,metric_type,date,totalLiquidityUSD,protocol
0,Ethereum-staking,tvl,2020-12-03 00:00:00,244094753,Aave
1,Ethereum-staking,tvl,2020-12-04 00:00:00,243189473,Aave
2,Ethereum-staking,tvl,2020-12-05 00:00:00,250327449,Aave
3,Ethereum-staking,tvl,2020-12-06 00:00:00,253742514,Aave
4,Ethereum-staking,tvl,2020-12-07 00:00:00,256062919,Aave
...,...,...,...,...,...
43314,Aptos-borrowed,tvl,2026-01-07 00:00:00,15965898,Aave
43315,Aptos-borrowed,tvl,2026-01-08 00:00:00,15977650,Aave
43316,Aptos-borrowed,tvl,2026-01-09 00:00:00,15998925,Aave
43317,Aptos-borrowed,tvl,2026-01-10 00:00:00,16014125,Aave


In [6]:
df_chain_tvls['protocol'] = 'Aave'

In [8]:
df_chain_tvls.to_csv('final_data.csv')